### Charger le Modèle Sauvegardé

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Charger le modèle et le tokenizer
model_path = "../model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Mettre le modèle en mode évaluation
model.eval()
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
model.to(device)


e:\Cours Télécom Paris\IADATA700 - Kik Big Data - Charlotte LACLAU\Projet-groupe\mangetamain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

### Faire des Prédictions

In [2]:
def predict_sentiment(text, model, tokenizer):
    """Prédit le sentiment d'un texte"""
    # Labels correspondant à votre entraînement
    label_map = {0: "Négatif", 1: "Neutre", 2: "Positif"}
    
    # Tokenization
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    # Déplacer sur le bon device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Prédiction
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        print(predictions)
        confidence = predictions[0][predicted_class].item()
    
    return label_map[predicted_class], confidence


In [4]:
# Tests de prédiction
test_reviews = [
    "Amazing! Will make again.",
    "Too salty.",
    "It was okay, nothing special.",
    "good"
]

for review in test_reviews:
    sentiment, confidence = predict_sentiment(review, model, tokenizer)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment} (Confiance: {confidence:.2%})", end="\n\n")

tensor([[0.2481, 0.0070, 0.7449]])
Review: Amazing! Will make again.
Sentiment: Positif (Confiance: 74.49%)

tensor([[0.4422, 0.5558, 0.0020]])
Review: Too salty.
Sentiment: Neutre (Confiance: 55.58%)

tensor([[0.1596, 0.8388, 0.0016]])
Review: It was okay, nothing special.
Sentiment: Neutre (Confiance: 83.88%)

tensor([[0.1806, 0.7518, 0.0676]])
Review: good
Sentiment: Neutre (Confiance: 75.18%)

